In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os, gc, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Cấu hình đường dẫn cơ sở
BASE_DIR = "/content/drive/MyDrive/Mining of massive dataset/preprocessing_output"
LOOKUP_PATH = os.path.join(BASE_DIR, "taxi_zone_lookup_grid.csv")
OUT_DIR = "/content/drive/MyDrive/Data_DL/processed_data/yellow_2024_full"

os.makedirs(OUT_DIR, exist_ok=True)

### tạo map LocationID -> (i,j)

In [ ]:
GRID_H, GRID_W = 10, 20
TIME_INTERVAL = "30min"  # Thay "30T" thành "30min" để tuân thủ chuẩn mới của Pandas
YEAR = 2024
DATASET = "yellow"

def get_month_paths(dataset: str, year: int, month: int):
    mm = f"{month:02d}"
    year_dir = os.path.join(BASE_DIR, f"{dataset}_data", f"{dataset}_{year}")
    vol_path = os.path.join(year_dir, f"{mm}_volume.csv")
    flow_path = os.path.join(year_dir, f"{mm}_flow.parquet")
    return vol_path, flow_path

In [ ]:
# 1. Tạo map LocationID -> (i, j)
grid = pd.read_csv(LOOKUP_PATH)
grid["Grid_X"] = grid["Grid_X"].astype(int)
grid["Grid_Y"] = grid["Grid_Y"].astype(int)

loc2ij = {int(row.LocationID): (int(row.Grid_X), int(row.Grid_Y)) for _, row in grid.iterrows()}
print("Grid_X range:", grid["Grid_X"].min(), grid["Grid_X"].max())
print("Grid_Y range:", grid["Grid_Y"].min(), grid["Grid_Y"].max())

# 2. Thu thập toàn bộ Time Index của 12 tháng
time_slots = set()
for month in range(1, 13):
    vol_path, _ = get_month_paths(DATASET, YEAR, month)
    if os.path.exists(vol_path):
        # Chỉ đọc cột time_bin để tiết kiệm RAM
        df_t = pd.read_csv(vol_path, usecols=["time_bin"])
        df_t["slot"] = pd.to_datetime(df_t["time_bin"], errors="coerce").dt.floor(TIME_INTERVAL)
        time_slots.update(df_t["slot"].dropna().unique())

time_index = sorted(list(time_slots))
time_map = {t: i for i, t in enumerate(time_index)}
T = len(time_index)

print(f"Tổng số time slots (1H/30min) cho cả năm {YEAR}: {T}")

Grid_X range: 0 9
Grid_Y range: 0 19
Tổng số time slots (1H/30min) cho cả năm 2024: 17586


In [ ]:
vol_memmap_path = os.path.join(OUT_DIR, "volume_temp.dat")
flow_memmap_path = os.path.join(OUT_DIR, "flow_temp.dat")

volume = np.memmap(vol_memmap_path, dtype=np.float32, mode='w+', shape=(T, GRID_H, GRID_W, 2))
flow = np.memmap(flow_memmap_path, dtype=np.float32, mode='w+', shape=(2, T, GRID_H, GRID_W, GRID_H, GRID_W))

print("Volume memmap shape:", volume.shape)
print("Flow memmap shape:", flow.shape)

Volume memmap shape: (17586, 10, 20, 2)
Flow memmap shape: (2, 17586, 10, 20, 10, 20)


### time_map từ volume

In [ ]:
for month in range(1, 13):
    vol_path, flow_path = get_month_paths(DATASET, YEAR, month)
    if not os.path.exists(vol_path) or not os.path.exists(flow_path):
        print(f"Bỏ qua tháng {month} do không tìm thấy file.")
        continue

    print(f"Processing Month {month}...")

    # ---------------- 1. XỬ LÝ VOLUME ----------------
    vol_df = pd.read_csv(vol_path)
    vol_df["slot"] = pd.to_datetime(vol_df["time_bin"], errors="coerce").dt.floor(TIME_INTERVAL)
    vol_df["LocationID"] = pd.to_numeric(vol_df["locationid"], errors="coerce")
    vol_df = vol_df.dropna(subset=["slot", "LocationID"])

    vol_hour = vol_df.groupby(["slot", "LocationID"], as_index=False)[["start_volume", "end_volume"]].sum()

    # Lọc và map indices
    valid_mask = vol_hour["LocationID"].isin(loc2ij) & vol_hour["slot"].isin(time_map)
    vol_valid = vol_hour[valid_mask].copy()

    coords = vol_valid["LocationID"].map(loc2ij)
    t_idx = vol_valid["slot"].map(time_map).values
    i_idx = np.array([c[0] for c in coords])
    j_idx = np.array([c[1] for c in coords])

    # Vectorized Add
    np.add.at(volume[..., 0], (t_idx, i_idx, j_idx), vol_valid["start_volume"].values)
    np.add.at(volume[..., 1], (t_idx, i_idx, j_idx), vol_valid["end_volume"].values)

    # Giải phóng bộ nhớ
    del vol_df, vol_hour, vol_valid, coords
    gc.collect()

    # ---------------- 2. XỬ LÝ FLOW ----------------
    pf = pq.ParquetFile(flow_path)
    flow_df = pf.read(columns=["time_bin", "pulocationid", "dolocationid", "flow_count"]).to_pandas()

    flow_df["slot"] = pd.to_datetime(flow_df["time_bin"], errors="coerce").dt.floor(TIME_INTERVAL)
    flow_df = flow_df.dropna(subset=["slot", "pulocationid", "dolocationid"])

    flow_hour = flow_df.groupby(["slot", "pulocationid", "dolocationid"], as_index=False)["flow_count"].sum()

    # Lọc và map indices
    valid_mask = (flow_hour["pulocationid"].isin(loc2ij) &
                  flow_hour["dolocationid"].isin(loc2ij) &
                  flow_hour["slot"].isin(time_map))
    flow_valid = flow_hour[valid_mask].copy()

    pu_coords = flow_valid["pulocationid"].map(loc2ij)
    do_coords = flow_valid["dolocationid"].map(loc2ij)

    t_idx = flow_valid["slot"].map(time_map).values
    i_idx = np.array([c[0] for c in pu_coords])
    j_idx = np.array([c[1] for c in pu_coords])
    k_idx = np.array([c[0] for c in do_coords])
    l_idx = np.array([c[1] for c in do_coords])
    counts = flow_valid["flow_count"].values

    # Timestep hiện tại (k=0)
    np.add.at(flow[0], (t_idx, i_idx, j_idx, k_idx, l_idx), counts)

    # Timestep trước đó (k=1)
    valid_prev = t_idx > 0
    np.add.at(flow[1], (t_idx[valid_prev] - 1, i_idx[valid_prev], j_idx[valid_prev], k_idx[valid_prev], l_idx[valid_prev]), counts[valid_prev])

    # Giải phóng bộ nhớ
    del flow_df, flow_hour, flow_valid, pu_coords, do_coords
    gc.collect()

print("Hoàn tất xử lý 12 tháng!")
print("Volume nonzero:", np.count_nonzero(volume))
print("Flow nonzero:", np.count_nonzero(flow))

Processing Month 1...
Processing Month 2...
Processing Month 3...
Processing Month 4...
Processing Month 5...
Processing Month 6...
Processing Month 7...
Processing Month 8...
Processing Month 9...
Processing Month 10...
Processing Month 11...
Processing Month 12...
Hoàn tất xử lý 12 tháng!
Volume nonzero: 1417810
Flow nonzero: 7833305


### khởi tạo volumn + flow

In [ ]:
# Xác định index cho Train và Test (24 ngày đầu tháng làm Train, còn lại làm Test)
time_series = pd.Series(time_index)
is_train_mask = time_series.dt.day <= 24

train_indices = np.where(is_train_mask)[0]
test_indices = np.where(~is_train_mask)[0]

# Chuyển dữ liệu từ memmap sang numpy array trên RAM để thao tác cuối cùng (hoặc lưu thẳng)
volume_train = np.array(volume[train_indices])
volume_test = np.array(volume[test_indices])
flow_train = np.array(flow[:, train_indices])
flow_test = np.array(flow[:, test_indices])

# Tính toán giá trị max trên tập Train để tránh Data Leakage
volume_train_max = volume_train.max()
flow_train_max = flow_train.max()

print("Volume_train_max:", volume_train_max)
print("Flow_train_max:", flow_train_max)

# Chuẩn hóa về [0, 1]
volume_train = volume_train / (volume_train_max + 1e-6)
volume_test = volume_test / (volume_train_max + 1e-6)
flow_train = flow_train / (flow_train_max + 1e-6)
flow_test = flow_test / (flow_train_max + 1e-6)

print("Đã hoàn tất chuẩn hóa.")
print(f"Volume Train Shape: {volume_train.shape}, Test Shape: {volume_test.shape}")
print(f"Flow Train Shape: {flow_train.shape}, Test Shape: {flow_test.shape}")

Volume_train_max: 1966.0
Flow_train_max: 707.0
Đã hoàn tất chuẩn hóa.
Volume Train Shape: (13833, 10, 20, 2), Test Shape: (3753, 10, 20, 2)
Flow Train Shape: (2, 13833, 10, 20, 10, 20), Test Shape: (2, 3753, 10, 20, 10, 20)


### group thành 1H

In [ ]:
# Lưu thành các file NPZ riêng biệt
np.savez_compressed(os.path.join(OUT_DIR, "volume_train.npz"), volume=volume_train)
np.savez_compressed(os.path.join(OUT_DIR, "volume_test.npz"), volume=volume_test)
np.savez_compressed(os.path.join(OUT_DIR, "flow_train.npz"), flow=flow_train)
np.savez_compressed(os.path.join(OUT_DIR, "flow_test.npz"), flow=flow_test)

print("Saved numpy files to:", OUT_DIR)

# Thiết lập file config data.json
config = {
  "volume_train": os.path.join(OUT_DIR, "volume_train.npz"),
  "volume_test": os.path.join(OUT_DIR, "volume_test.npz"),
  "flow_train": os.path.join(OUT_DIR, "flow_train.npz"),
  "flow_test": os.path.join(OUT_DIR, "flow_test.npz"),
  "volume_train_max": 1.0,
  "flow_train_max": 1.0,
  "timeslot_sec": 1800,
  "threshold": 0
}

json_path = os.path.join(OUT_DIR, "data.json")
with open(json_path, "w") as f:
    json.dump(config, f, indent=4)

print("Created config file:", json_path)

# (Tùy chọn) Xóa các file memmap tạm thời để tiết kiệm dung lượng Google Drive
try:
    del volume, flow
    os.remove(vol_memmap_path)
    os.remove(flow_memmap_path)
    print("Đã dọn dẹp các file tạm memmap.")
except Exception as e:
    print("Có lỗi trong quá trình dọn dẹp file tạm:", e)

Saved numpy files to: /content/drive/MyDrive/Data_DL/processed_data/yellow_2024_full
Created config file: /content/drive/MyDrive/Data_DL/processed_data/yellow_2024_full/data.json
Đã dọn dẹp các file tạm memmap.


### Fill flow từ flow.parquet

### Normalize